In this session you will:

- learn how to implement the Pagerank algorithm to determine the relative importance of webpages
- apply it to a large graph of links between Wikipedia articles

## 1. Pagerank on a toy graph

### Install fast graph analysis package (`igraph`)

In [ ]:
#!pip3 install igraph --user

### Create and visualize a toy graph

In [ ]:
from igraph import Graph, plot

def example_graph(edges):
    n = max(map(max, zip(*edges))) + 1
    vertex_names = ["v%i" % i for i in range(n)]
    print("%i vertices, %i edges" % (n, len(edges)))
    return vertex_names, edges

def to_igraph(vertex_names, edges):
    G = Graph(n=len(vertex_names), directed=True)
    G.add_edges(edges)
    G.vs["name"] = vertex_names
    return G

names, edges = example_graph([ (0, 0), (0, 2), (0, 3), (1, 0), (1, 3), (2, 1), (2, 3), (3, 1)])
G = to_igraph(names, edges)
plot(G, layout="fr", vertex_label=names, bbox=(400,400), margin=40)

### Display adjacency and transition matrices

In [ ]:
import numpy as np

def adjacency_matrix(edges, n):
    src, dest = zip(*edges)
    A = np.zeros((n, n))
    A[src, dest] = 1
    return A

# Compute transition matrix (row-stochastic): 
#    M[i, j] = probability of going from i to j in one step
# Note the result is undefined if there are nodes of out-degree 0, but these do not occur in our example
def transition_matrix(A):
    row_sums = A.sum(axis=1, keepdims=True)           # keepdims=True keeps the result as a column vector
    return A / row_sums
    
print("Adjacency matrix")
A = adjacency_matrix(edges, len(names))
print(A)

M = transition_matrix(A)
print("\nTransition matrix (row-normalized):")
print(M)

### Compute Pagerank vector and check that it satisfies the Pagerank linear equations

Recall that Pagerank is a solution to the recursive equation "a node is important iff it is pointed to by other important nodes". Each node distributes its "importance" (pagerank) equally among its outgoing neighbors. Thus, if $p$ is the Pagerank vector, $p_i$ should coincide with the importance received by node $i$, i.e., the $i$th row of $M^T \cdot p$. 

In [ ]:
pr = G.pagerank(damping=1)
print("Nodes by decreasing pagerank:")
for (p, name) in reversed(sorted(zip(pr, names))):
    print("%s: %lf" % (name, p))
    
print("\nsum(v) =", sum(pr))
print("Mᵀ · v =", M.T.dot(pr))
print("v =", pr)

# Note that we need to allow some tolerance when comparing floating-point values
print("max error =", np.abs(M.T.dot(pr) - pr).max())
assert np.allclose(M.T.dot(pr), pr)

### Alternatively, compute pagerank from eigenvector calculations (for didactic purposes)


In [ ]:
# The pagerank vector as the eigenvector of M.T associated to eigenvalue 1
def eig_pagerank(M):
    vals, vecs = np.linalg.eig(M.T)
    idx = np.argmin(np.abs(vals - 1))              # eigenvalue closest to 1; it should actually be 1    
    r = np.real(vecs[:, idx])    
    return r / r.sum()

r = eig_pagerank(M)    
print("PageRank vector:", r)
print("max error with previously computed pagerank =", np.abs(r - pr).max())

---

**Exercise 1:**  
Make sure you understand the code above. Now write a third way to compute the Pagerank by exploiting the random-surfer viewpoint of Pagerank. The idea is to compute the probability distribution of the surfer's location among all nodes in the graph after a large enough number of steps. Explain what interpretation the transpose of the matrix $M$ has in this setting, why we need to consider the transpose and why normalizing the adjacency matrix is necessary. 

Your function should take the matrix $M$, a number of steps $T$, and a column vector $v$ describing the initial probability distribution. For example, starting at a random node is represented by the vector `np.array([1, 1, 1, 1]) / 4` (if there are 4 nodes). Check that you still get the same pagerank vector as before.

Hint: ```np.linalg.matrix_power()```, ```np.ones()``` and/or ```np.zeros()``` might come in handy.

In [ ]:
def random_surfer_pagerank(M, T, v):
    return np.linalg.matrix_power(M.T, T).dot(v)

In [ ]:
v_init = np.ones(len(names)) / len(names)   # uniform initial distribution
pr_surfer = random_surfer_pagerank(M, 100, v_init)
print("PageRank vector:", pr_surfer)
print("max error with previously computed pagerank =", np.abs(pr_surfer - pr).max())

---

**Exercise 2:**  
In class we saw how introducing a damping factor $\lambda\in(0,1)$ allows us to ignore corner cases such as aperiodicity and lack of strong connectivity.
Modify your function from Exercise 1, as well as the eigenvalue-based solution, to incorporate a damping parameter and check that both still give the same result as `G.pagerank(damping=...)` from `igraph`.

In [ ]:
def random_surfer_pagerank_damping(M, T, v, d):
    n = M.shape[0]
    M_damped = d * M + (1 - d) * np.ones((n, n)) / n
    return np.linalg.matrix_power(M_damped.T, T).dot(v)

In [ ]:
v_init = np.ones(len(names)) / len(names)   # uniform initial distribution
d = 0.255
pr_damped = random_surfer_pagerank_damping(M, 100, v_init, d)
print("PageRank vector:", pr_damped, "(with damping d=%s)" % d)

pr = G.pagerank(damping=d)
print("PageRank vector:", np.round(pr, 8), "(from igraph)")


## 2. Pagerank on Wikipedia

Now we will write our own implementation of Pagerank (with a damping factor) on a real-world graph with a hundred million edges, built from hyperlinks in Wikipedia articles:

https://snap.stanford.edu/data/enwiki-2013.html

### Download and decompress dataset from the Stanford Network Analysis Project

In [ ]:
#!wget https://snap.stanford.edu/data/enwiki-2013.txt.gz && gunzip enwiki-2013.txt.gz
#!wget https://snap.stanford.edu/data/enwiki-2013-names.csv.gz && gunzip enwiki-2013-names.csv.gz

In [ ]:
import pandas as pd

# Load directed edges as (src, dest) pairs. Vertices are numbered from 0 to (number of vertices) - 1, inclusive.
src, dest = np.loadtxt('enwiki-2013.txt', comments='#', dtype=np.int32).T
edges = np.column_stack((src, dest))

# Load webpage titles
titles = pd.read_csv("enwiki-2013-names.csv", escapechar='\\', dtype={"node_id": "int32", "name": "string"})
titles = titles["name"].fillna("").to_numpy()           # replace missing titles with an empty string

n = max(len(titles), np.max(src) + 1, np.max(dest) + 1)
m = len(src)
print("Loaded graph: %i vertices, %i directed edges" % (n, m))

In [ ]:
# Create the sparse adjacency matrix A: A[src[i]][dest[i]] = 1 for all i in [m] and 0 elsewhere
# This is internally represented as an adjacency list mapping each vertex to its outgoing neighbors.
# So the space usage is O(m), rather than Omega(n^2), which would be impractical
from scipy import sparse
A = sparse.csr_matrix((np.ones_like(src, dtype=np.float32), (src, dest)), shape=(n, n), dtype=np.float32)

In [ ]:
# Note that some nodes have out-degree 0. You need to decide how to handle them.
outdeg = np.asarray(A.sum(axis=1)).ravel()
sinks = np.where(outdeg == 0)[0]
print("%i dead-end nodes" % len(sinks))

### Cheating by using libraries

#### You may use igraph it to verify the correctness of your code, once it has been implemented

In [ ]:
%%time
G = to_igraph(titles, edges)
pr_lib = G.pagerank(damping=0.8)

In [ ]:
import heapq

def show_topk(pr, titles, topk=20):
    print("Top %i nodes in order of decreasing pagerank:" % topk)
    for p, name in heapq.nlargest(topk, zip(pr, titles), key=lambda x: x[0]):
        print("%s: %lf" % (name, p))
        
show_topk(pr_lib, titles, 20)                

### Our problem

---

**Exercise 3:**  
Write an efficient implementation of Pagerank with a damping factor < 1 and compare with `igraph`'s implementation in terms of accuracy (e.g., difference in pagerank vectors, top $k$ overlap, etc.) and running time. You may use `numpy`, `scipy` and/or similar libraries for matrix operations if you wish, but no graph libraries are allowed.

It is essential to make use of the sparsity of this graph, so you will need to design an efficient solution in time and space usage. In particular, our earlier solution computing powers of $M^T$ cannot be used directly, because the powers of $M^T$ will be very dense.

Think about how we can implement a random step of the surfer efficiently, assuming we know the probability distribution of the surfer's location in the graph at the previous step. A single step corresponds to a multiplication between the matrix $M^T$ from above and the current pagerank vector, along with a correction to account for teleportation.

As a (relatively minor) implementation detail, you also need to decide on simple strategies to
- handle nodes with out-degree zero, and
- decide when to stop iterating.

In [ ]:
# v_t+1​=d⋅(M^T⋅v_t​)+(1−d​)/N x1+Corrección Sinks
def pagerank_manual(A, damping, tol, max_iter):
    n = A.shape[0]
    v = np.ones(n) / n
    t = (1-damping) / n # teleport
    outdeg = np.asarray(A.sum(axis=1)).ravel()
    sinks = np.where(outdeg == 0)[0] # Nodes sense sortida

    outdeg_aux = outdeg.copy()
    outdeg_aux[sinks] = 1        # per evitar divisió per zero

    D_inv = sparse.diags(1.0 / outdeg_aux)
    M_T = A.T.dot(D_inv)

    for it in range(max_iter):
        v_new = damping * M_T.dot(v) + t + damping * v[sinks].sum() / n
        err = np.linalg.norm(v_new - v, 1)
        v = v_new
        if err < tol:
            print("Converged in %i iterations" % (it+1))
            break
    return v

In [ ]:
import time

start_time = time.time()
my_pr = pagerank_manual(A, damping=0.8, tol=1e-6, max_iter=100)
end_time = time.time()
print(f"Tiempo de ejecución manual: {end_time - start_time:.2f} segundos")


pr_lib_arr = np.array(pr_lib) 
diff = np.sum(np.abs(my_pr - pr_lib_arr))
print(f"Diferencia total L1 con igraph: {diff:.2e}")


print("\n--- Top 20 PageRank Manual ---")
show_topk(my_pr, titles, 20)

---
**Exercise 4 (personalized Pagerank):**
Modify your solution from Exercise 3 to accept a teleportation set as input. Compute Pagerank again, but with teleportation reduced to the set of articles whose title contains the word "tennis". 

How do the results change and why? Can you find a top tennis-related article that is not in the teleportation set? If we decreased the damping parameter, would you expect to find more or fewer tennis-related articles among the top results?

In [ ]:
def pagerank_manual_teleportation(A, damping, tol, max_iter, teleport_set):
    n = A.shape[0]
    
    v_teleport = np.zeros(n, dtype=np.float32)
    v_teleport[teleport_set] = 1.0 / len(teleport_set)

    pr = np.ones(n, dtype=np.float32) / n
    
    outdeg = np.asarray(A.sum(axis=1)).ravel()
    sinks = np.where(outdeg == 0)[0]
    
    outdeg_safe = outdeg.copy()
    outdeg_safe[sinks] = 1
    
    D_inv = sparse.diags(1.0 / outdeg_safe)
    M_T = A.T.dot(D_inv)
    
    for it in range(max_iter):
        pr_old = pr.copy()
        sink_mass = damping * pr_old[sinks].sum()
        
        pr = damping * M_T.dot(pr_old) + sink_mass * v_teleport + (1 - damping) * v_teleport
        
        err = np.linalg.norm(pr - pr_old, 1)  
        
        if err < tol:
            print(f"Converged in {it+1} iterations (error: {err:.2e})")
            break
    
    pr = pr / pr.sum()
    
    return pr

In [ ]:
# 1. Identificar los índices de teleportación (Artículos que contienen "tennis")
# 'titles' es la lista de nombres que cargaste al principio del notebook
tennis_indices = [i for i, title in enumerate(titles) if 'tennis' in title.lower()]
print(f"Se encontraron {len(tennis_indices)} artículos relacionados con 'tennis'.")

# 2. Ejecutar tu función de PageRank Personalizado
# Usamos damping=0.85 (estándar) o 0.8 (si quieres consistencia con el anterior)
print("Calculando PageRank Personalizado...")
pr_tennis = pagerank_manual_teleportation(A, damping=0.8, tol=1e-6, max_iter=100, teleport_set=tennis_indices)

# 3. Mostrar los resultados Top 20
print("\n--- Top 20 PageRank Personalizado (Tennis) ---")
show_topk(pr_tennis, titles, 20)